# **📌 Token Classification using BERT**

This project demonstrates the implementation of a Token Classification model using BERT from the Transformers library.

The goal is to perform sequence labeling tasks such as Named Entity Recognition (NER) or Chunking.

The model is trained on the CoNLL dataset and evaluated using standard metrics like Precision, Recall, and F1-score. Proper preprocessing techniques such as tokenization, label alignment, and padding are applied to ensure accurate predictions.



In [12]:
!pip install -q transformers datasets seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


## 📦 Installing Required Libraries

In [13]:
import torch
import numpy as np

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, Trainer, TrainingArguments
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report

## 📂 Loading the Dataset (CoNLL-2003)

In [20]:
from datasets import load_dataset

dataset = load_dataset("conll2003", revision="refs/convert/parquet")

print(dataset)

conll2003/train/0000.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/312k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/283k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})


## 🔄 Tokenization and Label Alignment

In [21]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(examples["chunk_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

## 🏷️ Defining Label List for Token Classification

In [24]:
label_list = dataset["train"].features["chunk_tags"].feature.names
print("Chunk Labels:", label_list)

Chunk Labels: ['O', 'B-ADJP', 'I-ADJP', 'B-ADVP', 'I-ADVP', 'B-CONJP', 'I-CONJP', 'B-INTJ', 'I-INTJ', 'B-LST', 'I-LST', 'B-NP', 'I-NP', 'B-PP', 'I-PP', 'B-PRT', 'I-PRT', 'B-SBAR', 'I-SBAR', 'B-UCP', 'I-UCP', 'B-VP', 'I-VP']


## 🤖 Loading Pretrained BERT Model for Token Classification

In [25]:
from transformers import AutoModelForTokenClassification
import torch

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label_list),
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized be

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, el

## ⚙️ Training Arguments

In [26]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",   # ✅ important fix
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    fp16=True,
    logging_dir="./logs",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [27]:
import numpy as np
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_labels = [
        [label_list[l] for l in label if l != -100]
        for label in labels
    ]
    true_predictions = [
        [label_list[p] for (p, l) in zip(pred, label) if l != -100]
        for pred, label in zip(predictions, labels)
    ]

    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
    }

## 🚀 Training the Model

In [30]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
    data_collator=data_collator,   # ✅ THIS FIXES ERROR
)

In [31]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.456494,0.198582,0.909674,0.903913,0.906784
2,0.162437,0.183501,0.913819,0.911334,0.912575


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1756, training_loss=0.25557896868243035, metrics={'train_runtime': 185.725, 'train_samples_per_second': 151.202, 'train_steps_per_second': 9.455, 'total_flos': 680972157444126.0, 'train_loss': 0.25557896868243035, 'epoch': 2.0})

## 📊 Evaluating Model Performance

In [38]:
results = trainer.evaluate()
print("Evaluation Results:", results)

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
0,No log,0.185641,0.906952,0.909313,0.908131


Evaluation Results: {'eval_loss': 0.18564093112945557, 'eval_precision': 0.9069524104789956, 'eval_recall': 0.9093134397948479, 'eval_f1': 0.9081313905434509}


## 🧠 Performing Inference

In [36]:
sentence = "John works at Google in California"

tokens = sentence.split()
inputs = tokenizer(tokens, is_split_into_words=True, return_tensors="pt")

inputs = {k: v.to(device) for k, v in inputs.items()}

outputs = model(**inputs)
predictions = torch.argmax(outputs.logits, dim=2)

print("\nChunking Output:")
for word, pred in zip(tokens, predictions[0]):
    print(f"{word} → {label_list[pred.item()]}")


Chunking Output:
John → O
works → B-NP
at → B-VP
Google → B-PP
in → B-NP
California → B-PP


# **Task 7: Comparison**

POS Tagging vs Chunking

POS Tagging → assigns grammatical tags (noun, verb)
Chunking → groups words into phrases (NP, VP)

✔ POS = easier
✔ Chunking = more complex (context-based)


# **Task 8: Report / Blog**

## 🔹 Differences between POS Tagging and Chunking

* **POS Tagging** assigns grammatical labels to each word (noun, verb, etc.).
* **Chunking** groups words into meaningful phrases (noun phrase, verb phrase).
* POS → **word-level**, Chunking → **phrase-level**.



## 🔹 Challenges Faced

* Aligning labels with BERT subword tokens
* Handling special tokens using `-100`
* Dataset loading and version issues
* Padding errors (fixed using data collator)



## 🔹 Observations and Insights

* BERT captures context effectively
* Chunking gives deeper sentence understanding than POS
* Label alignment is very important
* Subword tokenization improves performance but adds complexity





# **✅ Conclusion**

In this project, we successfully implemented a token classification model using BERT for tasks like Named Entity Recognition and Chunking. The model was trained on the CoNLL dataset with proper preprocessing techniques such as tokenization, label alignment, and dynamic padding.

The model achieved a strong performance with high precision, recall, and F1-score, demonstrating the effectiveness of transformer-based architectures in sequence labeling tasks.

This project provided practical insights into handling real-world NLP challenges and highlighted the importance of preprocessing and evaluation techniques in building robust models.